In [ ]:
from google.colab import drive
drive.mount('/content/mydrive')

In [2]:
import os
repo_path = '/content/optimized-summarization'
if not os.path.exists(repo_path):
    !git clone https://github.com/srinisvas/optimized-summarization.git
else:
    print("Repo already exists, skipping clone.")

# Check files inside
os.listdir(repo_path)

Cloning into 'optimized-summarization'...
remote: Enumerating objects: 1625, done.
remote: Counting objects: 100% (50/50), done.
remote: Compressing objects: 100% (37/37), done.
remote: Total 1625 (delta 35), reused 13 (delta 13), pack-reused 1575 (from 2)
Receiving objects: 100% (1625/1625), 108.00 MiB | 28.45 MiB/s, done.
Resolving deltas: 100% (463/463), done.


['optimized-summarization', '.git', '.idea', 'README.md']

In [1]:
!pip install -U transformers accelerate bitsandbytes

import os
import json
import time
import torch
import requests
from typing import Dict, Any, Optional
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

# --- MODEL CONFIGURATION ---
MODEL_NAME = "HuggingFaceTB/SmolLM3-3B"

# Initialize model and tokenizer (using 4-bit quantization to fit on most Colab GPUs)
try:
    print(f"Loading Model: {MODEL_NAME} with 4-bit quantization...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

    # Use BitsAndBytesConfig for 4-bit quantization
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16,
    )

    model = AutoModelForCausalLM.from_pretrained(
        MODEL_NAME,
        device_map="auto",
        quantization_config=bnb_config,
    )
    print("Model loaded successfully. Ready for inference.")
except Exception as e:
    print(f"Could not load the model. Ensure you have a GPU runtime and all libraries are installed.")
    print(f"Error details: {e}")
    # Exit or raise error if model loading fails
    raise

# Set max retries for the generation loop, mostly for edge cases
MAX_RETRIES = 3



# 1. Directory containing the original, normalized papers
NORMALIZED_DIR = '/content/optimized-summarization/optimized-summarization/Normalized-papers'

# 2. Output folder for the final summaries
OUTPUT_DIR = '/content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprompts/'

os.makedirs(OUTPUT_DIR, exist_ok=True)


# --- LLM Generation Function ---

def call_local_llm(chat_messages: list, max_retries: int = MAX_RETRIES) -> Optional[str]:

    #Generates text locally using the loaded  model .

    # 1. template to turn the list of messages into a single prompt string
    prompt = tokenizer.apply_chat_template(chat_messages, tokenize=False, add_generation_prompt=True)

    for attempt in range(max_retries):
        try:
            # 2. Tokenize and generate
            input_ids = tokenizer(prompt, return_tensors="pt").to(model.device)

            # Start the generation process
            outputs = model.generate(
                **input_ids,
                max_new_tokens=512,
                temperature=0.3,
                do_sample=True, # Use sampling based on temperature
                pad_token_id=tokenizer.eos_token_id
            )

            # 3. Decode the output and clean the text
            generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

            # outputs the full prompt + response. We must strip the prompt.
            clean_text = generated_text.replace(prompt, "").strip()

            if "Executive Summary" in clean_text:
                # Ensure we start exactly from the summary title
                clean_text = clean_text.split("Executive Summary", 1)[-1].strip()
                return "Executive Summary\n\n" + clean_text

            # If the specific start phrase isn't found, return the cleaned output anyway
            return clean_text.strip()

        except Exception as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                print(f"   [Attempt {attempt + 1}] Local generation failed ({type(e).__name__}). Retrying in {wait_time}s...")
                time.sleep(wait_time)
            else:
                print(f"   [Attempt {attempt + 1}] Local generation failed. Max retries reached. Skipping document.")
                return None
    return None

def load_paper_data(file_name: str) -> Optional[Dict[str, Any]]:
    """
    Ablation Study Modification: Only loads data from the Normalized directory.
    Aggregates all sections into a single 'full_text' field.
    """
    base_id = file_name.replace('.json', '')
    normalized_path = os.path.join(NORMALIZED_DIR, file_name)

    try:
        with open(normalized_path, 'r', encoding='utf-8') as f:
            paper_data = json.load(f)

        # Aggregate every section into one continuous text block
        all_sections = []
        for section in paper_data.get('sections', []):
            title = section.get('title', 'Untitled Section').strip()
            content = section.get('content', '').strip()
            if content:
                all_sections.append(f"### {title}\n{content}")

        full_text = "\n\n".join(all_sections)

        if not full_text:
            print(f"   [Warning] {file_name} appears to be empty. Skipping.")
            return None

        return {
            "paper_id": base_id,
            "full_text": full_text
        }

    except FileNotFoundError:
        print(f"   [Error] File not found: {normalized_path}")
        return None
    except Exception as e:
        print(f"   [Error] Skipping {file_name} due to unexpected error: {e}")
        return None

# --- Prompt Engineering ---

# --- Updated One-Shot Example for Tech Journalist ---
# --- Updated One-Shot Example (still grounded in a paper, just different persona) ---
ONE_SHOT_EXAMPLE = """
**[Example] Title: The Role of Quantum Computing in Differential Privacy in IoT Networks**

This paper examines challenges in maintaining privacy while sharing high-utility data in IoT networks. It proposes a quantum-resistant differential privacy framework combining homomorphic encryption and quantum-hardened noise protocols. Experiments show that the approach reduces computational overhead and improves query response time by 35% compared to traditional methods, balancing privacy protection with data utility. The framework can help meet regulatory compliance requirements without compromising analytical value. Future work will focus on hardware integration and testing against real quantum attack scenarios.
"""

def create_adaptive_prompt(data: Dict[str, Any]) -> list:
    """
    Constructs the final prompt using the full paper content as context.
    Updated role/persona and instructions, but all output grounded in the paper.
    """

    # 1. System Instruction (Updated Role/Persona)
    system_instruction = (
        "You are an expert reviewer who condenses complex research papers into concise summaries. "
        "Your task is to capture the main research contributions, methodology, and results exactly as they appear in the paper. "
        "Do NOT include interpretations, opinions, or information not present in the provided text."
    )

    # 2. Updated Instructions (different from previous CoT)
    cot_instruction = (
        "\n\n**Processing Instructions:**\n"
        "1. **Identify Core Contributions:** Focus on the key problems addressed and the solutions proposed.\n"
        "2. **Highlight Methods and Results:** Clearly state the methodology and outcomes as described in the paper.\n"
        "3. **Stay Grounded:** Do not add any external information or assumptions. Use only what is in the provided text.\n"
        "4. **Use Example as Style:** Follow the flow and tone of the provided one-shot example."
    )

    context_injection = f"""
**Target Paper: {data['paper_id']}**
{cot_instruction}

--- ONE-SHOT EXAMPLE (For style and tone only) ---
{ONE_SHOT_EXAMPLE}
--- END OF EXAMPLE ---

**Full Paper Content:**
{data['full_text']}

---
**TASK:** Generate a summary based strictly on the content above. Begin your response with the title: **Executive Summary**. Return only the summary text.
"""

    # 3. Chat Messages List
    chat_messages = [
        {"role": "system", "content": system_instruction},
        {"role": "user", "content": context_injection}
    ]
    return chat_messages



# --- Main Orchestration Loop ---

def run_orchestration():
    """
    Iterates through all papers, loads data, prompts LLM, and saves results.
    """

    print(f"Starting Adaptive Summarization with CoT using Local Model: {MODEL_NAME}.")

    processed_count = 0

    try:
        file_list = sorted(os.listdir(NORMALIZED_DIR))
    except FileNotFoundError:
        print(f"\n[FATAL ERROR] Cannot find the main directory: {NORMALIZED_DIR}. Please check the path.")
        return

    for file_name in file_list:
        if file_name.endswith(".json"):
            paper_id = file_name.replace('.json', '')
            print(f"\n--- Orchestrating Summary for {paper_id} ---")

            data = load_paper_data(file_name)

            if data is None:
                continue

            # This creates the list of messages for the chat template
            chat_messages = create_adaptive_prompt(data)

            print("   Generating summary locally on GPU...")
            summary_text = call_local_llm(chat_messages)

            if summary_text:
                output_file_name = f"{paper_id}.txt"
                output_path = os.path.join(OUTPUT_DIR, output_file_name)

                try:
                    with open(output_path, 'w', encoding='utf-8') as f:
                        f.write(summary_text)
                    print(f"   SUCCESS! Summary saved to: {output_path}")
                    processed_count += 1
                except Exception as e:
                    print(f"   [Error] Failed to save output file: {e}")
            else:
                print(f"   [Error] Failed to generate summary for {paper_id}.")

    print(f"\n--- Pipeline execution finished for all documents. Generated {processed_count} final summaries. ---")
    print(f"Check your '{OUTPUT_DIR}' folder in Google Drive!")

if __name__ == "__main__":
    run_orchestration()







Loading Model: HuggingFaceTB/SmolLM3-3B with 4-bit quantization...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/4.97G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/1.18G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

Model loaded successfully. Ready for inference.
Starting Adaptive Summarization with CoT using Local Model: HuggingFaceTB/SmolLM3-3B.

--- Orchestrating Summary for A Bibliometric View of AI Ethics Development ---
   Generating summary locally on GPU...
   SUCCESS! Summary saved to: /content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprompts/A Bibliometric View of AI Ethics Development.txt

--- Orchestrating Summary for A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI ---
   Generating summary locally on GPU...
   SUCCESS! Summary saved to: /content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprompts/A Model for Using Ethical Theory to Specify Epistemic Goals for Explainable AI.txt

--- Orchestrating Summary for A Privacy Impact Assessment Tool for Cloud Computing ---
   Generating summary locally on GPU...
   SUCCESS! Summary saved to: /content/mydrive/MyDrive/NLP Project/ablation_adaptive prompting_differentprom

KeyboardInterrupt: 